# Дашборд конверсий

Первичный анализ визитов и регистраций за период с 2023-03-01 по 2023-09-01.

In [ ]:
import requests
import pandas as pd

BEGIN = '2023-03-01'
END = '2023-09-01'
API = 'https://data-charts-api.hexlet.app'

visits = pd.DataFrame(requests.get(f'{API}/visits', params={'begin': BEGIN, 'end': END}).json())
registrations = pd.DataFrame(requests.get(f'{API}/registrations', params={'begin': BEGIN, 'end': END}).json())
visits['datetime'] = pd.to_datetime(visits['datetime'])
registrations['datetime'] = pd.to_datetime(registrations['datetime'])
visits = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)]
visits = visits.sort_values('datetime').drop_duplicates('visit_id', keep='last')
visits.head(), registrations.head()

In [ ]:
visits.describe(include='all')
registrations.describe(include='all')

In [ ]:
daily_visits = visits.assign(date_group=visits['datetime'].dt.date).groupby(['date_group', 'platform']).size().rename('visits')
daily_registrations = registrations.assign(date_group=registrations['datetime'].dt.date).groupby(['date_group', 'platform']).size().rename('registrations')
conversion = pd.concat([daily_visits, daily_registrations], axis=1).fillna(0).reset_index()
conversion['conversion'] = conversion['registrations'] / conversion['visits'] * 100
conversion = conversion.sort_values(['date_group', 'platform'])
conversion.to_json('./conversion.json')
conversion.head(10)

In [ ]:
platform_funnel = visits.groupby('platform').size().rename('visits').to_frame()
platform_funnel['registrations'] = registrations.groupby('platform').size()
platform_funnel = platform_funnel.fillna(0)
platform_funnel['conversion_rate'] = platform_funnel['registrations'] / platform_funnel['visits'] * 100
platform_funnel.sort_values('conversion_rate', ascending=False)